In [2]:
import pandas as pd

df = pd.read_csv(r"D:\Моя папка\AB_Test\ab_test_results.csv")

df.head()

,user_id,variant,converted,session_duration,pages_visited,age
0,1,control,0,3.494324,7,57
1,2,treatment,0,4.452306,4,24
2,3,control,0,4.384545,1,49
3,4,control,0,6.338822,6,36
4,5,control,0,1.576521,3,23


In [3]:
# Кількість рядків і стовпців
print("Розмір таблиці:", df.shape)

# Типи даних і кількість заповнених значень
df.info()

# Пропуски в кожному стовпці
print("\nПропуски:")
print(df.isna().sum())

# Повністю однакові рядки
print("\nДублікати рядків:", df.duplicated().sum())

# Повторні ID користувачів
print("Повторні user_id:", df["user_id"].duplicated().sum())

Розмір таблиці: (10000, 6)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   user_id           10000 non-null  int64  
 1   variant           10000 non-null  object 
 2   converted         10000 non-null  int64  
 3   session_duration  10000 non-null  float64
 4   pages_visited     10000 non-null  int64  
 5   age               10000 non-null  int64  
dtypes: float64(1), int64(4), object(1)
memory usage: 468.9+ KB

Пропуски:
user_id             0
variant             0
converted           0
session_duration    0
pages_visited       0
age                 0
dtype: int64

Дублікати рядків: 0
Повторні user_id: 0


In [4]:
# Назви груп і кількість користувачів у кожній
print("Групи тестування:")
print(df["variant"].value_counts())

# Значення конверсії
print("\nЗначення converted:")
print(df["converted"].value_counts())

# Основні характеристики числових показників
df[["session_duration", "pages_visited", "age"]].describe()

Групи тестування:
variant
control      5013
treatment    4987
Name: count, dtype: int64

Значення converted:
converted
0    8751
1    1249
Name: count, dtype: int64


,session_duration,pages_visited,age
count,10000.000000,10000.000000,10000.000000
mean,6.021124,3.990500,40.825200
std,2.453395,2.222364,13.432524
min,1.000000,0.000000,18.000000
25%,4.290201,2.000000,30.000000
50%,5.867869,4.000000,41.000000
75%,7.649210,5.000000,53.000000
max,15.230423,14.000000,64.000000


In [5]:
# Кількість користувачів і конверсій у кожній групі
conversion_summary = df.groupby("variant").agg(
    users=("user_id", "count"),
    conversions=("converted", "sum")
)

conversion_summary

,users,conversions
variant,,
control,5013,534
treatment,4987,715


In [6]:
# Конверсія у відсотках
conversion_summary["conversion_rate_pct"] = (
    conversion_summary["conversions"]
    / conversion_summary["users"]
    * 100
)

conversion_summary.round(2)

,users,conversions,conversion_rate_pct
variant,,,
control,5013,534,10.65
treatment,4987,715,14.34


In [7]:
# Середня тривалість сесії у хвилинах
session_summary = df.groupby("variant").agg(
    avg_session_duration=("session_duration", "mean")
)

session_summary.round(2)

,avg_session_duration
variant,
control,5.03
treatment,7.02


In [8]:
# Середня кількість переглянутих сторінок
pages_summary = df.groupby("variant").agg(
    avg_pages_visited=("pages_visited", "mean")
)

pages_summary.round(2)

,avg_pages_visited
variant,
control,2.98
treatment,5.01


In [9]:
from statsmodels.stats.proportion import proportions_ztest

# Кількість конверсій: control, treatment
conversions = conversion_summary.loc[
    ["control", "treatment"], "conversions"
]

# Кількість користувачів у тому самому порядку
users = conversion_summary.loc[
    ["control", "treatment"], "users"
]

# Двосторонній z-тест
z_stat, p_value = proportions_ztest(
    count=conversions,
    nobs=users,
    alternative="two-sided"
)

print(f"Z-статистика: {z_stat:.4f}")
print(f"P-value: {p_value:.6g}")

if p_value < 0.05:
    print("Відхиляємо H₀: різниця конверсій статистично значуща.")
else:
    print("Недостатньо підстав відхилити H₀.")

Z-статистика: -5.5730
P-value: 2.50321e-08
Відхиляємо H₀: різниця конверсій статистично значуща.


In [10]:
from scipy.stats import ttest_ind

# Розділяємо дані на дві групи
control = df[df["variant"] == "control"]
treatment = df[df["variant"] == "treatment"]

# T-тест Велча для тривалості сесії
t_stat_session, p_value_session = ttest_ind(
    control["session_duration"],
    treatment["session_duration"],
    equal_var=False
)

print(f"T-статистика: {t_stat_session:.4f}")
print(f"P-value: {p_value_session:.6g}")

if p_value_session < 0.05:
    print("Відхиляємо H₀: різниця середньої тривалості сесії статистично значуща.")
else:
    print("Недостатньо підстав відхилити H₀.")

T-статистика: -44.3410
P-value: 0
Відхиляємо H₀: різниця середньої тривалості сесії статистично значуща.


In [11]:
# T-тест Велча для кількості переглянутих сторінок
t_stat_pages, p_value_pages = ttest_ind(
    control["pages_visited"],
    treatment["pages_visited"],
    equal_var=False
)

print(f"T-статистика: {t_stat_pages:.4f}")
print(f"P-value: {p_value_pages:.6g}")

if p_value_pages < 0.05:
    print("Відхиляємо H₀: різниця середньої кількості сторінок статистично значуща.")
else:
    print("Недостатньо підстав відхилити H₀.")

T-статистика: -51.2558
P-value: 0
Відхиляємо H₀: різниця середньої кількості сторінок статистично значуща.


In [12]:
# Створюємо вікові групи
df["age_group"] = pd.cut(
    df["age"],
    bins=[0, 25, 35, 45, float("inf")],
    labels=["<25", "25-34", "35-44", "45+"],
    right=False
)

# Перевіряємо кількість користувачів у кожній групі
df["age_group"].value_counts(sort=False)

age_group
<25      1453
25-34    2220
35-44    2173
45+      4154
Name: count, dtype: int64

In [13]:
# Показники за віковими групами та варіантами тесту
age_summary = df.groupby(
    ["age_group", "variant"],
    observed=True
).agg(
    users=("user_id", "count"),
    conversions=("converted", "sum"),
    avg_session_duration=("session_duration", "mean"),
    avg_pages_visited=("pages_visited", "mean")
)

# Конверсія у відсотках
age_summary["conversion_rate_pct"] = (
    age_summary["conversions"]
    / age_summary["users"]
    * 100
)

age_summary.round(2)

users  conversions  avg_session_duration  \
age_group variant                                               
<25       control      738           73                  5.06   
          treatment    715          105                  6.98   
25-34     control     1079          122                  5.02   
          treatment   1141          152                  7.08   
35-44     control     1075          111                  5.03   
          treatment   1098          170                  7.05   
45+       control     2121          228                  5.02   
          treatment   2033          288                  6.98   

                     avg_pages_visited  conversion_rate_pct  
age_group variant                                            
<25       control                 2.95                 9.89  
          treatment               4.93                14.69  
25-34     control                 2.98                11.31  
          treatment               4.84                13.32  
35-44     control                 2.96                10.33  
          treatment               5.05                15.48  
45+       control                 3.00                10.75  
          treatment               5.10                14.17

In [14]:
# Список для збереження результатів
age_conversion_results = []

# Повторюємо z-тест для кожної вікової групи
for age_group in df["age_group"].cat.categories:
    segment = df[df["age_group"] == age_group]

    segment_control = segment[segment["variant"] == "control"]
    segment_treatment = segment[segment["variant"] == "treatment"]

    z_stat_age, p_value_age = proportions_ztest(
        count=[
            segment_control["converted"].sum(),
            segment_treatment["converted"].sum()
        ],
        nobs=[
            len(segment_control),
            len(segment_treatment)
        ],
        alternative="two-sided"
    )

    age_conversion_results.append({
        "age_group": age_group,
        "z_stat": z_stat_age,
        "p_value": p_value_age
    })

age_conversion_tests = pd.DataFrame(age_conversion_results)

age_conversion_tests

,age_group,z_stat,p_value
0,<25,-2.786257,0.005332
1,25-34,-1.442557,0.149145
2,35-44,-3.582022,0.000341
3,45+,-3.337425,0.000846


In [15]:
from statsmodels.stats.multitest import multipletests

# Поправка Холма для чотирьох тестів конверсії
significant, p_adjusted, _, _ = multipletests(
    age_conversion_tests["p_value"],
    alpha=0.05,
    method="holm"
)

# Додаємо результати до таблиці
age_conversion_tests["p_value_holm"] = p_adjusted
age_conversion_tests["significant_holm"] = significant

age_conversion_tests.round(6)

,age_group,z_stat,p_value,p_value_holm,significant_holm
0,<25,-2.786257,0.005332,0.010664,True
1,25-34,-1.442557,0.149145,0.149145,False
2,35-44,-3.582022,0.000341,0.001364,True
3,45+,-3.337425,0.000846,0.002537,True


In [16]:
age_session_results = []

for age_group in df["age_group"].cat.categories:
    segment = df[df["age_group"] == age_group]

    segment_control = segment[segment["variant"] == "control"]
    segment_treatment = segment[segment["variant"] == "treatment"]

    t_stat_age, p_value_age = ttest_ind(
        segment_control["session_duration"],
        segment_treatment["session_duration"],
        equal_var=False
    )

    age_session_results.append({
        "age_group": age_group,
        "t_stat": t_stat_age,
        "p_value": p_value_age
    })

age_session_tests = pd.DataFrame(age_session_results)

# Поправка Холма
significant, p_adjusted, _, _ = multipletests(
    age_session_tests["p_value"],
    alpha=0.05,
    method="holm"
)

age_session_tests["p_value_holm"] = p_adjusted
age_session_tests["significant_holm"] = significant

age_session_tests

,age_group,t_stat,p_value,p_value_holm,significant_holm
0,<25,-15.948079,1.687590e-52,1.687590e-52,True
1,25-34,-21.643890,4.052320e-94,1.215696e-93,True
2,35-44,-21.176192,2.218134e-90,4.436268e-90,True
3,45+,-28.183866,3.759318e-159,1.503727e-158,True


In [17]:
age_pages_results = []

for age_group in df["age_group"].cat.categories:
    segment = df[df["age_group"] == age_group]

    segment_control = segment[segment["variant"] == "control"]
    segment_treatment = segment[segment["variant"] == "treatment"]

    t_stat_age, p_value_age = ttest_ind(
        segment_control["pages_visited"],
        segment_treatment["pages_visited"],
        equal_var=False
    )

    age_pages_results.append({
        "age_group": age_group,
        "t_stat": t_stat_age,
        "p_value": p_value_age
    })

age_pages_tests = pd.DataFrame(age_pages_results)

# Поправка Холма для чотирьох тестів кількості сторінок
significant, p_adjusted, _, _ = multipletests(
    age_pages_tests["p_value"],
    alpha=0.05,
    method="holm"
)

age_pages_tests["p_value_holm"] = p_adjusted
age_pages_tests["significant_holm"] = significant

age_pages_tests

,age_group,t_stat,p_value,p_value_holm,significant_holm
0,<25,-19.568732,3.196405e-75,3.196405e-75,True
1,25-34,-22.543544,3.178916e-101,6.357831e-101,True
2,35-44,-24.712419,3.752039e-118,1.125612e-117,True
3,45+,-33.630526,6.215750e-217,2.486300e-216,True


In [18]:
# Об'єднуємо три таблиці за назвою групи
overall_summary = (
    conversion_summary
    .join(session_summary)
    .join(pages_summary)
)

overall_summary.round(2)

,users,conversions,conversion_rate_pct,avg_session_duration,avg_pages_visited
variant,,,,,
control,5013,534,10.65,5.03,2.98
treatment,4987,715,14.34,7.02,5.01


In [19]:
# Загальні результати статистичних тестів
overall_tests = pd.DataFrame({
    "metric": [
        "conversion_rate",
        "session_duration",
        "pages_visited"
    ],
    "test": [
        "Two-proportion z-test",
        "Welch t-test",
        "Welch t-test"
    ],
    "statistic": [
        z_stat,
        t_stat_session,
        t_stat_pages
    ],
    "p_value": [
        p_value,
        p_value_session,
        p_value_pages
    ]
})

# Висновок за рівня значущості 0,05
overall_tests["significant"] = overall_tests["p_value"] < 0.05

# Зручний запис p-value для дашборда
overall_tests["p_value_display"] = overall_tests["p_value"].apply(
    lambda p: "<0.001" if p < 0.001 else f"{p:.4f}"
)

overall_tests

,metric,test,statistic,p_value,significant,p_value_display
0,conversion_rate,Two-proportion z-test,-5.573048,2.503207e-08,True,<0.001
1,session_duration,Welch t-test,-44.340981,0.000000e+00,True,<0.001
2,pages_visited,Welch t-test,-51.255843,0.000000e+00,True,<0.001


In [20]:
from pathlib import Path

# Папка для результатів
output_folder = Path(r"D:\Моя папка\AB_Test\results")
output_folder.mkdir(parents=True, exist_ok=True)

# Загальні показники та показники за віком
overall_summary.reset_index().to_csv(
    output_folder / "overall_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

age_summary.reset_index().to_csv(
    output_folder / "age_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

# Результати статистичних тестів
test_tables = {
    "overall_tests": overall_tests,
    "age_conversion_tests": age_conversion_tests,
    "age_session_tests": age_session_tests,
    "age_pages_tests": age_pages_tests
}

for filename, table in test_tables.items():
    table.to_csv(
        output_folder / f"{filename}.csv",
        index=False,
        encoding="utf-8-sig"
    )

print("Файли збережено в:", output_folder)

Файли збережено в: D:\Моя папка\AB_Test\results
